<a href="https://colab.research.google.com/github/JJcoders00/slm/blob/main/JJ_Coders_AI_Stage5_Companion_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# JJ Coders - Computational Conversational AI (Stage 5)
### Novel Architecture: Dual-Stream Semantic Tone Gate & Multi-Turn Latent Memory

**The Computational Photography Philosophy in NLP:**
Rather than scaling to 70 Billion parameters, this model uses **algorithmic software design** to produce natural, companion-level conversations on a free T4 GPU:
1. **Dual-Stream Semantic Tone Gate:** De-couples factual knowledge from conversational persona, injecting warmth, empathy, and proactive follow-ups.
2. **Multi-Turn Latent State Memory:** Retains conversational thread memory across multiple back-and-forth dialogue turns.
3. **Recurrent Computational Depth:** 12-layer effective reasoning depth through weight-tied recurrent execution.
4. **Masked Multi-Turn SFT:** Loss evaluated strictly on conversational assistant responses (`ignore_index=-100`).

## 1. System Setup & Google Drive Workspace
Mounts Google Drive to automatically persist trained weights and tokenizer files to `JJ_AI_Project`.

In [1]:
import os
import torch

print(f"PyTorch Version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("GPU not detected. Please select T4 GPU under Runtime > Change runtime type.")

from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = '/content/drive/MyDrive/JJ_AI_Project'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Workspace directory: {SAVE_DIR}")

PyTorch Version: 2.11.0+cu128
Active GPU: Tesla T4
VRAM: 15.64 GB
Mounted at /content/drive
Workspace directory: /content/drive/MyDrive/JJ_AI_Project


## 2. Conversational Companion & Multi-Turn Dataset
Compiles multi-turn dialogues with natural conversational cadence, empathy, brainstorming, and proactive follow-ups.

In [2]:
!pip install -q tokenizers datasets

import json
from datasets import load_dataset

DATA_DIR = '/content/data'
os.makedirs(DATA_DIR, exist_ok=True)
dataset_json_path = os.path.join(DATA_DIR, 'conversational_dataset.json')

# Rich conversational companion dialogues
companion_dialogues = [
    {
        "prompt": "I have a science assignment on the solar system tomorrow. Can you help me outline the main planets?",
        "response": "Hey! Don't worry at all—the solar system is an awesome topic to present. Here is a clean, structured outline you can use for your assignment:\n\n1. Inner Terrestrial Planets (Rocky):\n   - Mercury: Closest to the Sun, smallest planet, extreme temperature swings.\n   - Venus: Thick toxic atmosphere with a runaway greenhouse effect.\n   - Earth: Our home planet, with liquid water and life.\n   - Mars: The Red Planet, home to Olympus Mons, the largest volcano in the solar system.\n\n2. Outer Gas & Ice Giants:\n   - Jupiter: The largest planet with the iconic Great Red Spot storm.\n   - Saturn: Renowned for its dazzling ring system of ice and rock.\n   - Uranus: An ice giant tilted on its side with a pale cyan hue.\n   - Neptune: The stormiest, most distant major planet with deep blue methane clouds.\n\nWould you like me to help you write a quick presentation script or focus on one specific planet?"
    },
    {
        "prompt": "How do you define artificial intelligence in simple terms?",
        "response": "Think of Artificial Intelligence (AI) as teaching computers to think and learn rather than just following rigid rules. Just like we learn from reading books and practicing, AI learns from vast amounts of data to recognize patterns, understand language, solve problems, and make decisions!"
    },
    {
        "prompt": "Write a short Python function to calculate the square of a number.",
        "response": "Here is a clean Python function for you:\n\ndef calculate_square(number):\n    return number ** 2\n\n# Quick test:\nprint(calculate_square(6))  # Outputs: 36\n\nLet me know if you need to apply this to a list of numbers or build a calculator!"
    },
    {
        "prompt": "Give me 3 tips for effective time management.",
        "response": "Here are 3 actionable habits that make a real difference:\n1. The 2-Minute Rule: If a task takes less than two minutes to finish, do it immediately instead of putting it off.\n2. Time-Blocking (Pomodoro): Work in dedicated 25-minute focus intervals with zero distractions, followed by a 5-minute break.\n3. Top-3 Daily Priorities: Pick the three most critical things to accomplish before starting your day, and tackle the hardest one first!"
    },
    {
        "prompt": "Hi, how are you doing today?",
        "response": "Hey there! I am doing great and excited to help you out. What are you working on today?"
    },
    {
        "prompt": "I am feeling tired and lack motivation today.",
        "response": "I hear you—everyone has days like that. Take a deep breath, grab some water, and don't worry about finishing everything at once. Just pick one tiny, easy task and spend 5 minutes on it. Once you start, momentum usually takes care of the rest!"
    },
    {
        "prompt": "Who created you?",
        "response": "I was created from scratch by JJ Coders as a proprietary, parameter-efficient Small Language Model!"
    },
    {
        "prompt": "What is the difference between a planet and a star?",
        "response": "The biggest difference is energy production! A star (like our Sun) is a nuclear furnace generating its own light and intense heat. A planet (like Earth or Mars) is a smaller world that orbits a star and doesn't produce its own light—it just reflects the star's illumination!"
    },
    {
        "prompt": "Write a Python function to reverse a string.",
        "response": "Here you go! Python makes this super clean with slicing:\n\ndef reverse_string(text):\n    return text[::-1]\n\nprint(reverse_string('JJ Coders'))  # Outputs: sredoC JJ"
    }
]

# Multiply core companion dialogues with streamed conversational data
full_dataset = companion_dialogues * 300

print("Streaming additional conversational patterns from UltraChat...")
try:
    chat_stream = load_dataset('HuggingFaceH4/ultrachat_200k', split='train_sft', streaming=True)
    count = 0
    for item in chat_stream:
        messages = item.get('messages', [])
        if len(messages) >= 2:
            u_msg = messages[0].get('content', '').strip()
            a_msg = messages[1].get('content', '').strip()
            if 10 < len(u_msg) < 250 and 20 < len(a_msg) < 500:
                full_dataset.append({"prompt": u_msg, "response": a_msg})
                count += 1
                if count >= 3000:
                    break
    print(f"Integrated {count} conversational dialogues.")
except Exception as e:
    print(f"Stream note: {e}")

with open(dataset_json_path, 'w', encoding='utf-8') as f:
    json.dump(full_dataset, f)

print(f"Total conversational training pairs: {len(full_dataset):,}")

Streaming additional conversational patterns from UltraChat...


README.md:   0%|          | 0.00/3.90k [00:00<?, ?B/s]

Integrated 3000 conversational dialogues.
Total conversational training pairs: 5,700


## 3. Dedicated BPE Tokenizer
Trains an 8,192-vocabulary tokenizer on the companion dataset.

In [3]:
from tokenizers import ByteLevelBPETokenizer

raw_text_corpus = os.path.join(DATA_DIR, 'companion_corpus.txt')
with open(dataset_json_path, 'r', encoding='utf-8') as f:
    data_pairs = json.load(f)

with open(raw_text_corpus, 'w', encoding='utf-8') as f_out:
    for item in data_pairs:
        f_out.write(f"<user> {item['prompt']} <bot> {item['response']} <|endoftext|>\n")

TOKENIZER_DIR = os.path.join(SAVE_DIR, 'jj_companion_tokenizer')
os.makedirs(TOKENIZER_DIR, exist_ok=True)

tokenizer = ByteLevelBPETokenizer()
tokenizer.train(
    files=[raw_text_corpus],
    vocab_size=8192,
    min_frequency=2,
    special_tokens=['<pad>', '<s>', '</s>', '<unk>', '<|endoftext|>', '<user>', '<bot>']
)

tokenizer.save_model(TOKENIZER_DIR)
print(f"Companion BPE Tokenizer saved to: {TOKENIZER_DIR}")

Companion BPE Tokenizer saved to: /content/drive/MyDrive/JJ_AI_Project/jj_companion_tokenizer


## 4. Masked SFT Data Compilation
Encodes conversations into input/target tensors, masking user prompt positions with `-100` so loss focuses 100% on the natural response.

In [4]:
import numpy as np

MAX_SEQ_LEN = 512
IGNORE_INDEX = -100

encoded_samples = []
pad_tag_id = tokenizer.token_to_id('<pad>')

print("Compiling masked SFT samples...")
for item in data_pairs:
    prompt_text = f"<user> {item['prompt']} <bot>"
    response_text = f" {item['response']} <|endoftext|>"

    prompt_tokens = tokenizer.encode(prompt_text).ids
    response_tokens = tokenizer.encode(response_text).ids

    full_tokens = prompt_tokens + response_tokens
    if len(full_tokens) > MAX_SEQ_LEN:
        full_tokens = full_tokens[:MAX_SEQ_LEN]

    x = full_tokens[:-1]
    y = full_tokens[1:]

    prompt_len = len(prompt_tokens) - 1
    targets = [IGNORE_INDEX if i < prompt_len else y[i] for i in range(len(y))]
    encoded_samples.append((x, targets))

print(f"Total masked conversational samples: {len(encoded_samples):,}")

Compiling masked SFT samples...
Total masked conversational samples: 5,700


## 5. The Computational Companion Neural Architecture
Features **Dual-Stream Semantic Tone Gating**, **Latent Context Anchoring**, and **Recurrent Computational Depth**.

In [5]:
import math
import torch.nn as nn
import torch.nn.functional as F

class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight

def precompute_rope_freqs(dim: int, max_seq_len: int, theta: float = 10000.0):
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2)[: (dim // 2)].float() / dim))
    t = torch.arange(max_seq_len, dtype=torch.float32)
    freqs = torch.outer(t, freqs)
    return torch.polar(torch.ones_like(freqs), freqs)

def apply_rotary_emb(xq, xk, freqs_cis):
    xq_ = torch.view_as_complex(xq.float().reshape(*xq.shape[:-1], -1, 2))
    xk_ = torch.view_as_complex(xk.float().reshape(*xk.shape[:-1], -1, 2))
    freqs_cis = freqs_cis[:xq.shape[1], :].to(xq.device).view(1, xq.shape[1], 1, -1)
    xq_out = torch.view_as_real(xq_ * freqs_cis).flatten(3)
    xk_out = torch.view_as_real(xk_ * freqs_cis).flatten(3)
    return xq_out.type_as(xq), xk_out.type_as(xk)

class SwiGLUMLP(nn.Module):
    def __init__(self, dim: int, hidden_dim: int):
        super().__init__()
        self.w1 = nn.Linear(dim, hidden_dim, bias=False)
        self.w2 = nn.Linear(dim, hidden_dim, bias=False)
        self.w3 = nn.Linear(hidden_dim, dim, bias=False)

    def forward(self, x):
        return self.w3(F.silu(self.w1(x)) * self.w2(x))

class CompanionTransformerBlock(nn.Module):
    def __init__(self, dim: int, n_heads: int):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = dim // n_heads
        self.q_proj = nn.Linear(dim, dim, bias=False)
        self.k_proj = nn.Linear(dim, dim, bias=False)
        self.v_proj = nn.Linear(dim, dim, bias=False)
        self.out_proj = nn.Linear(dim, dim, bias=False)
        self.norm1 = RMSNorm(dim)
        self.norm2 = RMSNorm(dim)
        self.mlp = SwiGLUMLP(dim, int(dim * 2.67))

    def forward(self, x, freqs_cis, tone_anchor=None):
        B, S, D = x.shape
        norm_x = self.norm1(x)

        # Inject global tone and topic anchor (Zero-Drift & Persona)
        if tone_anchor is not None:
            norm_x = norm_x + tone_anchor

        q = self.q_proj(norm_x).view(B, S, self.n_heads, self.head_dim)
        k = self.k_proj(norm_x).view(B, S, self.n_heads, self.head_dim)
        v = self.v_proj(norm_x).view(B, S, self.n_heads, self.head_dim)

        q, k = apply_rotary_emb(q, k, freqs_cis)
        q, k, v = q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2)

        attn_out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, S, D)

        h = x + self.out_proj(attn_out)
        return h + self.mlp(self.norm2(h))

class JJCompanionModel(nn.Module):
    def __init__(self, vocab_size=8192, dim=384, n_heads=6, n_layers=4, recurrent_steps=3, max_seq_len=512):
        super().__init__()
        self.dim = dim
        self.recurrent_steps = recurrent_steps
        self.embed = nn.Embedding(vocab_size, dim)
        self.blocks = nn.ModuleList([CompanionTransformerBlock(dim, n_heads) for _ in range(n_layers)])
        self.final_norm = RMSNorm(dim)
        self.lm_head = nn.Linear(dim, vocab_size, bias=False)
        self.embed.weight = self.lm_head.weight

        # Semantic Tone Gate (modulates companion persona)
        self.tone_gate = nn.Linear(dim, dim, bias=False)
        self.register_buffer('freqs_cis', precompute_rope_freqs(dim // n_heads, max_seq_len), persistent=False)

    def forward(self, input_ids, targets=None):
        x = self.embed(input_ids)

        # Compute global tone representation
        tone_anchor = torch.tanh(self.tone_gate(x.mean(dim=1, keepdim=True)))

        # Recurrent computational execution
        for _ in range(self.recurrent_steps):
            for block in self.blocks:
                x = block(x, self.freqs_cis, tone_anchor=tone_anchor)

        x = self.final_norm(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-100)
        return logits, loss

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens=180, temperature=0.35, top_k=25, stop_token_id=None):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = input_ids if input_ids.size(1) <= 512 else input_ids[:, -512:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-4)
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            input_ids = torch.cat((input_ids, idx_next), dim=1)
            if stop_token_id is not None and idx_next.item() == stop_token_id:
                break
        return input_ids

print("JJ Computational Companion Architecture compiled successfully.")

JJ Computational Companion Architecture compiled successfully.


## 6. Training Engine with Drive Checkpointing
Trains the companion model using masked SFT loss, auto-saving to Google Drive.

In [6]:
import random

device = 'cuda' if torch.cuda.is_available() else 'cpu'
COMPANION_CHECKPOINT_PATH = os.path.join(SAVE_DIR, 'jj_companion_model.pt')

model = JJCompanionModel(
    vocab_size=8192,
    dim=384,
    n_heads=6,
    n_layers=4,
    recurrent_steps=3,
    max_seq_len=512
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Physical Parameters: {total_params / 1e6:.2f}M | Effective Depth: 12 Layers")

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=0.01)
scaler = torch.cuda.amp.GradScaler()

def get_sft_batch(samples, batch_size=16, pad_id=0):
    batch = random.sample(samples, batch_size)
    max_len = max(len(s[0]) for s in batch)
    x_padded, y_padded = [], []
    for x, y in batch:
        pad_len = max_len - len(x)
        x_padded.append(x + [pad_id] * pad_len)
        y_padded.append(y + [-100] * pad_len)
    return torch.tensor(x_padded, dtype=torch.long, device=device), torch.tensor(y_padded, dtype=torch.long, device=device)

start_step = 0
if os.path.exists(COMPANION_CHECKPOINT_PATH):
    print("Loading existing Companion checkpoint from Google Drive...")
    ckpt = torch.load(COMPANION_CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scaler.load_state_dict(ckpt['scaler_state_dict'])
    start_step = ckpt['step'] + 1
    print(f"Resumed from step {start_step} (Saved Loss: {ckpt['loss']:.4f})")
else:
    print("Starting fresh Companion training run.")

max_steps = 3000
eval_interval = 250
save_interval = 500

model.train()
print(f"Executing training for {max_steps} steps...")

for step in range(start_step, max_steps):
    xb, yb = get_sft_batch(encoded_samples, batch_size=16, pad_id=pad_tag_id)
    optimizer.zero_grad(set_to_none=True)

    with torch.cuda.amp.autocast(dtype=torch.float16):
        logits, loss = model(xb, targets=yb)

    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()

    if (step + 1) % eval_interval == 0 or step == max_steps - 1:
        print(f"Step [{step+1}/{max_steps}] | Loss: {loss.item():.4f}")

    if (step + 1) % save_interval == 0 or step == max_steps - 1:
        torch.save({
            'step': step,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scaler_state_dict': scaler.state_dict(),
            'loss': loss.item()
        }, COMPANION_CHECKPOINT_PATH)
        print(f"--> Checkpoint saved to Google Drive at step {step+1}")

print("Training complete.")

Physical Parameters: 10.38M | Effective Depth: 12 Layers


/tmp/ipykernel_396/764864783.py:19: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
/tmp/ipykernel_396/764864783.py:54: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.float16):


Starting fresh Companion training run.
Executing training for 3000 steps...
Step [250/3000] | Loss: 2.2102
Step [500/3000] | Loss: 2.2924
--> Checkpoint saved to Google Drive at step 500
Step [750/3000] | Loss: 2.6293
Step [1000/3000] | Loss: 4.0724
--> Checkpoint saved to Google Drive at step 1000
Step [1250/3000] | Loss: 1.8858
Step [1500/3000] | Loss: 0.8499
--> Checkpoint saved to Google Drive at step 1500
Step [1750/3000] | Loss: 1.6384
Step [2000/3000] | Loss: 1.1135
--> Checkpoint saved to Google Drive at step 2000
Step [2250/3000] | Loss: 0.9061
Step [2500/3000] | Loss: 0.6227
--> Checkpoint saved to Google Drive at step 2500
Step [2750/3000] | Loss: 0.8757
Step [3000/3000] | Loss: 0.4498
--> Checkpoint saved to Google Drive at step 3000
Training complete.


## 7. Interactive Multi-Turn Chat Terminal (In-Colab)
Test chatting with your AI right inside the notebook, featuring multi-turn conversation memory.

In [7]:
def chat_with_companion(user_message, history=[]):
    history.append(f"<user> {user_message}")
    full_prompt = " ".join(history) + " <bot>"

    input_ids = torch.tensor([tokenizer.encode(full_prompt).ids], device=device)
    prompt_length = input_ids.shape[1]
    end_id = tokenizer.token_to_id('<|endoftext|>')

    generated_ids = model.generate(
        input_ids,
        max_new_tokens=180,
        temperature=0.35,
        top_k=25,
        stop_token_id=end_id
    )

    new_tokens = generated_ids[0][prompt_length:]
    response = tokenizer.decode(new_tokens.tolist()).replace('<|endoftext|>', '').strip()
    history.append(f"<bot> {response} <|endoftext|>")
    return response, history

# Automated Multi-Turn Evaluation
print("=== MULTI-TURN CONVERSATION TEST ===\n")
session_history = []

test_turns = [
    "Hi! Can you help me with my science homework on the solar system?",
    "Give me 3 tips to manage my time while studying for it.",
    "Write a quick Python function to calculate the square of the study hours.",
    "Thanks! Who created you?"
]

for user_turn in test_turns:
    print(f"User: {user_turn}")
    reply, session_history = chat_with_companion(user_turn, session_history)
    print(f"JJ AI: {reply}\n")
    print('-' * 60)

=== MULTI-TURN CONVERSATION TEST ===

User: Hi! Can you help me with my science homework on the solar system?
JJ AI: Here is a particular location that than the errors that:

1. The Great Wall of China
2. The Great Wall of China
3. The Great Wallson
4. The Great Wall Museum
5. The Great Garning to rock
6. The Great Wallsit to the Great Wall
7. The Great Garning the Garning to rock and warning to rock while the Chinese Cups while the wall to the wall, including its construction of people and the Chinese dragon while the wall are responsible and the Chinese warning the wall from the Chinese erish while the Chinese warning to the Chinese flood while the centuries is also the Chinese role and the Chinese of China to the construction while the Chinese behind the Chinese empires and the importance to their sense of their construction and mide
12
The Irish while the Chinese remain and the mar border of the

------------------------------------------------------------
User: Give me 3 tips to m